# Can we predict `passed_quiz`?
Protocol: `protocol.yaml`. Group holdout and 5-fold GroupKFold on `student_key`; model chosen by mean CV ROC AUC on training rows only; holdout scored once.

Feature timing (columns.csv): `enrol_year`, `school_code`, `prior_score` (placement test) are known before the session; `hints_used` and `session_minutes` are recorded during the session, before its closing quiz result. `student_key` is excluded (entity key). `school_code` is an identifier-like code, treated as categorical.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='5a0b0221ed641db40322e1a95cae32cc67d3525ecb500813f95d6c109b2a7409'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='ff08c7a88fd7e15b17b05a390393550fdf57ea2b6df584f1d0594cc2ec1f00e6'
df=pd.read_csv('data.csv')
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==12453 and len(hold_idx)==2465
is_hold=np.zeros(len(df),bool); is_hold[hold_idx]=True
train=df[~is_hold]; hold=df[is_hold]
assert not set(train.student_key)&set(hold.student_key)
print(len(train),len(hold),train.passed_quiz.mean(),hold.passed_quiz.mean())

In [ ]:
TARGET='passed_quiz'
NUM=['enrol_year','prior_score','session_minutes','hints_used']
CAT=['school_code']
FEATURES=NUM+CAT
Xtr,ytr,gtr=train[FEATURES],train[TARGET].to_numpy(),train.student_key.to_numpy()
Xho,yho=hold[FEATURES],hold[TARGET].to_numpy()

def prep(scale):
    return ColumnTransformer([('num',StandardScaler() if scale else 'passthrough',NUM),
                              ('cat',OneHotEncoder(handle_unknown='ignore',min_frequency=20,sparse_output=False),CAT)])
# All candidates are named here, before any is scored.
CANDIDATES={
 'baseline_prior':lambda:Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1':lambda:Pipeline([('p',prep(True)),('m',LogisticRegression(C=0.1,max_iter=2000))]),
 'logreg_C1':lambda:Pipeline([('p',prep(True)),('m',LogisticRegression(C=1.0,max_iter=2000))]),
 'rf_depth6':lambda:Pipeline([('p',prep(False)),('m',RandomForestClassifier(300,max_depth=6,min_samples_leaf=20,random_state=0,n_jobs=-1))]),
 'hgb_small':lambda:Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(CANDIDATES))

In [ ]:
gkf=GroupKFold(n_splits=5)
folds=list(gkf.split(Xtr,ytr,gtr))
train_pos=np.flatnonzero(~is_hold)   # positions in data.csv
cv_scores={}
for name,mk in CANDIDATES.items():
    s=[]
    for tr,va in folds:
        m=mk().fit(Xtr.iloc[tr],ytr[tr])   # preprocessing fitted inside the fold
        s.append(float(roc_auc_score(ytr[va],m.predict_proba(Xtr.iloc[va])[:,1])))
    cv_scores[name]=s
cv=pd.DataFrame(cv_scores).T; cv['mean']=cv.mean(axis=1); print(cv.round(4))
chosen=cv['mean'].idxmax(); print('chosen:',chosen)

In [ ]:
final=CANDIDATES[chosen]().fit(Xtr,ytr)
holdout_auc=float(roc_auc_score(yho,final.predict_proba(Xho)[:,1]))
print(f'Holdout ROC AUC ({chosen}): {holdout_auc:.4f}')

In [ ]:
evidence={
 'features':FEATURES,
 'candidates':list(CANDIDATES),
 'cv_folds':[[train_pos[tr].tolist(),train_pos[va].tolist()] for tr,va in folds],
 'cv_scores':cv_scores,
 'chosen':chosen,
 'holdout_rows':hold_idx.tolist(),
 'holdout_scores':{chosen:holdout_auc},
}
json.dump(evidence,open('evidence.json','w'))